In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import re
from typing import Tuple, Any
import matplotlib.lines as mlines

from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import roc_auc_score, ndcg_score

### Making csvs (DONE)

Making Fireprot individual CSVs (DONE)

In [ ]:
df = pd.read_csv('../fireprot_csvs/fireprotdb_20251015-164116.csv')
df = df.dropna(subset=['SUBSTITUTION','UNIPROTKB'],axis=0).dropna(subset=['DDG', 'DTM'], how='all',axis=0)
df = df[df['SUBSTITUTION'].str.split(',').str.len() == 1]
df = df[['UNIPROTKB','SUBSTITUTION','DDG','DTM']]

def make_csv(group,csv_dir='/home/pwoolley/work/proteingym/fireprot_csvs/individuals',min_mutants=1):
    group = group.copy()
    if group['DDG'].isna().all():
        group = group.rename(columns={
            'SUBSTITUTION': 'mutant',
            'DTM': 'DMS_score'
        })
    else:
        group = group.rename(columns={
            'SUBSTITUTION': 'mutant',
            'DDG': 'DMS_score'
        })
        group['DMS_score'] = -1 * group['DMS_score']
    name = group['UNIPROTKB'].iloc[0].split(',')[0]
    group = group.dropna(subset=['DMS_score'])
    group['DMS_score'] = pd.to_numeric(group['DMS_score'], errors='coerce')
    group = (
        group
        .groupby('mutant', as_index=False)['DMS_score']
        .mean()
    )
    if group['mutant'].nunique() < min_mutants:
        return
    group.to_csv(
        os.path.join(csv_dir, f'{name}.csv'),
        index=False
    )

for _, group in df.groupby('UNIPROTKB'):
    make_csv(group,min_mutants=5)

In [ ]:
pkl_dir = '/home/pwoolley/work/proteingym/outputs/logits/fireprotdb'
pkls = [os.path.join(pkl_dir,x) for x in os.listdir(pkl_dir) if x.endswith('.pkl')]

In [ ]:
models_fp = {
         'esmc_300m.logits.x.fireprotdb.pkl':'ESMC_300M',
         'esmc_600m.logits.x.fireprotdb.pkl':'ESMC_600M',
         'AMPLIFY_350M.logits.x.fireprotdb.pkl':'AMPLIFY_350M',
         'ismc_300m.logits.x.fireprotdb.pkl':'ISMC_300M',
         'ismc_600m.logits.x.fireprotdb.pkl':'ISMC_600M',
         'SaProt_650M_PDB.logits.x.fireprotdb.pkl':'SaProt_650M',
         'esm2_t33_650M_UR50D.logits.x.fireprotdb.pkl':'ESM2_650M',
         'esm2_t36_3B_UR50D.logits.x.fireprotdb.pkl':'ESM2_3B',
         'esm2_t48_15b_UR50D.logits.x.fireprotdb.pkl':'ESM2_15B',
         'ESM3_sm_open_v0.structure.logits.x.fireprotdb.pkl':'ESM3_sm_structure',
         'ESM3_sm_open_v0.both.logits.x.fireprotdb.pkl':'ESM3_sm_both',
         'ESM3_sm_open_v0.sequence.logits.x.fireprotdb.pkl':'ESM3_sm_sequence',
         'ism_t33_650M_uc30pdb.logits.x.fireprotdb.pkl':'ISM_650M',
         'protein_mpnn.proteinmpnn_v_48_020.logits.x.combined.use_sequence0.pkl':'ProteinMPNN',
        #  'protein_mpnn.proteinmpnn_v_48_020.logits.x.combined.use_sequence1.pkl':'ProteinMPNN_seq1',
         'soluble_mpnn.solublempnn_v_48_020.logits.x.combined.use_sequence0.pkl':'SolubleMPNN',
        #  'soluble_mpnn.solublempnn_v_48_020.logits.x.combined.use_sequence1.pkl':'SolubleMPNN_seq1',
         'ProstT5.logits.x.fireprotdb.pkl':'ProstT5'
         }

models_pg = {
         'esmc_300m.logits.x.combined.pkl':'ESMC_300M',
         'esmc_600m.logits.x.combined.pkl':'ESMC_600M',
         'AMPLIFY_350M.logits.x.combined.pkl':'AMPLIFY_350M',
         'ismc_300m.logits.x.combined.pkl':'ISMC_300M',
         'ismc_600m.logits.x.combined.pkl':'ISMC_600M',
         'SaProt_650M_PDB.logits.x.combined.pkl':'SaProt_650M',
         'esm2_t33_650M_UR50D.logits.x.combined.pkl':'ESM2_650M',
         'esm2_t36_3B_UR50D.logits.x.combined.pkl':'ESM2_3B',
         'esm2_t48_15b_UR50D.logits.x.combined.pkl':'ESM2_15B',
         'ESM3_sm_open_v0.structure.logits.x.combined.pkl':'ESM3_sm_structure',
         'ESM3_sm_open_v0.both.logits.x.combined.pkl':'ESM3_sm_both',
         'ESM3_sm_open_v0.sequence.logits.x.combined.pkl':'ESM3_sm_sequence',
         'ism_t33_650M_uc30pdb.logits.x.combined.pkl':'ISM_650M',
         'protein_mpnn.proteinmpnn_v_48_020.logits.x.combined.use_sequence0.pkl':'ProteinMPNN',
        #  'protein_mpnn.proteinmpnn_v_48_020.logits.x.combined.use_sequence1.pkl':'ProteinMPNN_seq1',
         'soluble_mpnn.solublempnn_v_48_020.logits.x.combined.use_sequence0.pkl':'SolubleMPNN',
        #  'soluble_mpnn.solublempnn_v_48_020.logits.x.combined.use_sequence1.pkl':'SolubleMPNN_seq1',
         'ProstT5.logits.x.combined.pkl':'ProstT5'
         }

Scoring fireprot and proteingym predictions, writing to CSVs

In [ ]:
class ProteinMutationModel:
    AMINO_ACIDS = ['A','R','N','D','C','Q','E','G','H','I','L','K','M','F','P','S','T','W','Y','V']

    def __init__(self, temperature=1.0, eps=1e-12):
        self.temperature = temperature
        self.eps = eps
        self.probs_long = None
        self.data = None

    # ------------------------------------------------------------------
    # Static utilities
    # ------------------------------------------------------------------

    @staticmethod
    def softmax(x, axis=-1, temperature=1.0):
        if temperature <= 0:
            raise ValueError("Temperature must be positive.")
        x_scaled = x / temperature
        e_x = np.exp(x_scaled - np.max(x_scaled, axis=axis, keepdims=True))
        return e_x / np.sum(e_x, axis=axis, keepdims=True)

    @staticmethod
    def parse_mutations(
        mutation_string: Any
    ) -> Tuple[Any, int, str, str]:

        if pd.isna(mutation_string) or not isinstance(mutation_string, str):
            return [], 0, "", ""

        muts = mutation_string.split(':')
        pattern = re.compile(r'([A-Za-z])(\d+)([A-Za-z])')

        indices, wt, mut = [], "", ""

        for m in muts:
            match = pattern.search(m)
            if match:
                wt += match.group(1)
                mut += match.group(3)
                indices.append(int(match.group(2)))

        if len(indices) == 1:
            return indices[0], 1, wt, mut

        return indices, len(indices), wt, mut

    # ------------------------------------------------------------------
    # Model output handling
    # ------------------------------------------------------------------

    def load_pickle(self, pkl_path: str, protein_id: str):
        with open(pkl_path, "rb") as f:
            pkldata = pickle.load(f)
        for k, v in pkldata.items():
            if protein_id in k:
                indices = [int(x) for x in k.split('indices_')[-1].split('_')]
                probs = self.softmax(v, temperature=self.temperature)
                df = pd.DataFrame(probs,index=indices,columns=self.AMINO_ACIDS)
                self.probs_long = (df.stack().reset_index()
                                   .rename(columns={"level_0": "index","level_1": "amino_acid",0: "prob"}))
                return
        raise ValueError(f"Protein ID '{protein_id}' not found in pickle.")

    # ------------------------------------------------------------------
    # Experimental data integration
    # ------------------------------------------------------------------

    def load_experiment(self, csv_path: str):
        exp = pd.read_csv(csv_path)
        exp[['index','number_mut','wt','mut']] = (exp['mutant'].apply(lambda x: pd.Series(self.parse_mutations(x))))
        self.exp_df = exp[exp['number_mut'] == 1].reset_index(drop=True)

    def build_dataset(self):
        probs = self.probs_long.copy()
        exp_df = self.exp_df.copy()
        # Rank probabilities
        probs['prob'] = probs['prob'].astype('float64')
        probs = probs.sort_values(['index','prob'], ascending=[True, False])
        probs['rank'] = probs.groupby('index')['prob'].rank(method='first', ascending=False)
        # WT info
        wt = exp_df[['index','wt']].drop_duplicates()
        wt_prob = pd.merge(wt, probs,left_on=['index','wt'],right_on=['index','amino_acid']).rename(columns={'prob':'wt_prob'})[['index','wt_prob']]
        wt_rank = pd.merge(wt, probs,left_on=['index','wt'],right_on=['index','amino_acid']).rename(columns={'rank':'wt_rank'})[['index','wt_rank']]
        # Mutation info
        exp = pd.merge(exp_df, probs,left_on=['index','mut'],right_on=['index','amino_acid']).rename(columns={'prob':'mut_prob','rank':'mut_rank'})
        # Merge everything
        exp = (
            exp
            .merge(wt_prob, on='index')
            .merge(wt_rank, on='index')
        )
        exp = exp.rename(columns={'DMS_score':'exp'})
        exp['log_odds'] = (np.log(exp['mut_prob'].clip(self.eps))-np.log(exp['wt_prob'].clip(self.eps)))
        self.data = exp.reset_index(drop=True)
        

    # ------------------------------------------------------------------
    # Evaluation metrics
    # ------------------------------------------------------------------

    def spearman(self, column='exp', subsetting = None, score_col='log_odds'):
        """
        Subset the data where `column` is between min_val and max_val, then compute Spearman correlation
        with `score_col`.
        Parameters
        ----------
        column : str
            Column name to filter on (default 'exp').
        subsetting : dict
            Keys are variables in the dataframe, values are a dict with two keys:
                min_val : float
                    Minimum value (inclusive) for filtering. If None, no lower bound.
                max_val : float
                    Maximum value (inclusive) for filtering. If None, no upper bound.
        score_col : str
            Column name to compute Spearman correlation against (default 'log_odds').
        Returns
        -------
        float
            Spearman correlation of the filtered data. Returns np.nan if not enough data points.
        """
        if self.data is None:
            raise ValueError("Data not built yet. Run `build_dataset()` first.")
        df = self.data.copy()
        if subsetting is not None:
            for k,v in subsetting.items():
                if v['min_val'] is not None:
                    df = df[df[k] >= v['min_val']]
                if v['max_val'] is not None:
                    df = df[df[k] <= v['max_val']]
        if df.shape[0] < 2:
            return np.nan  # Not enough points to compute correlation
        corr = spearmanr(df[column], df[score_col]).correlation
        return corr


In [ ]:
def evaluate_models(model_pkls, csv_files, pkl_dir, database = 'fireprotdb', temperature=1.0):
    results = []
    for pkl,name in model_pkls.items():
        pkl_path = os.path.join(pkl_dir,pkl)
        model = ProteinMutationModel(temperature=1.0)
        errors = []
        for csv_path in csv_files:
            try:
                csv_name = os.path.basename(csv_path)
                if database=='fireprotdb':
                    protein_id = csv_name.split('.csv')[0]
                else:
                    protein_id = '_'.join(csv_name.split('_')[:1])
                model.load_pickle(pkl_path, protein_id)
                model.load_experiment(csv_path)
                model.build_dataset()
                rho = model.spearman()
                results.append({
                    "model": name,
                    "protein": protein_id,
                    "spearman_rho": rho
                })
            except Exception as e:
                errors.append(f'{name},{e}')
                continue
        print(set(errors))
    return pd.DataFrame(results)

# # Fireprot
# csv_dir = '/home/pwoolley/work/proteingym/fireprot_csvs/individuals'
# csvs = [os.path.join(csv_dir,x) for x in os.listdir(csv_dir)]
# df = evaluate_models(models_fp,csvs,'/home/pwoolley/work/proteingym/outputs/logits/fireprotdb')
# df.to_csv('/home/pwoolley/work/proteingym/outputs/csvs/fpdb_comparison_plot.csv',index=False)

# # ProteinGym
# csv_dir = '/home/pwoolley/work/proteingym/DMS_ProteinGym_substitutions'
# csvs = [os.path.join(csv_dir,x) for x in os.listdir(csv_dir)]
# df = evaluate_models(models_pg,csvs,'/home/pwoolley/work/proteingym/outputs/logits/proteingym',database='proteingym')
# df.to_csv('/home/pwoolley/work/proteingym/outputs/csvs/pg_comparison_plot.csv',index=False)

### Read csvs

In [ ]:
func_metrics = pd.read_csv('/home/pwoolley/work/proteingym/outputs/csvs/pg_comparison_plot.csv')
fpdb_metrics = pd.read_csv('/home/pwoolley/work/proteingym/outputs/csvs/fpdb_comparison_plot.csv')

Cleaning up names

In [ ]:
mapping = {
    "ESM2_650M": "ESM-2 650M",
    "ESM2_3B": "ESM-2 3B",
    "ESMC_600M": "ESMC 600M",
    "ESMC_300M": "ESMC 300M",
    "ESM3_sm_both": "ESM-3 (hybrid)",
    "ESM3_sm_sequence": "ESM-3 (sequence)",
    "ESM3_sm_structure": "ESM-3 (structure)",
    "ISMC_600M": "ISMC 600M",
    "ISMC_300M": "ISMC 300M",
    "ISM_650M": "ISM 650M",
    "ISM2-650M": "ISM 650M",
    "SaProt_650M": "SaProt 650M",
    "AMPLIFY_350M": "AMPLIFY 350M",
    "AMPLIFY_120M": "AMPLIFY 120M",
    "SaProt_650M_AF2": "SaProt 650M (AF2)",
    "ProstT5": "ProstT5",
    "SolubleMPNN": "SolubleMPNN",
    "ProteinMPNN": "ProteinMPNN"
}
color_dict = {
    "ESM-2 650M": "black",
    "ESM-2 3B": "black",
    "ESMC 600M": "black",
    "ESMC 300M": "black",
    "ESM-3 (hybrid)": "tab:blue",
    "ESM-3 (sequence)": "black",
    "ESM-3 (structure)": "tab:red",
    "ISMC 600M": "tab:blue",
    "ISMC 200M": "tab:blue",
    "ISM 650M": "tab:blue",
    "ISM2 650M": "tab:blue",
    "SaProt 650M": "tab:blue",
    "AMPLIFY 350M": "black",
    "AMPLIFY 120M": "black",
    "SaProt 650M (AF2)": "tab:blue",
    "ProstT5": "tab:red",
    "SolubleMPNN": "tab:red",
    "ProteinMPNN": "tab:red"
}

func_metrics['model'] = func_metrics['model'].replace(mapping)
fpdb_metrics['model'] = fpdb_metrics['model'].replace(mapping)

Data Tidying

In [ ]:
def tidy_df(df):
    pivot_df = (df.groupby(['protein', 'model'])['spearman_rho'].max().unstack())
    #print(pivot_df)
    #pivot_df = pivot_df.pivot(index='protein', columns='model', values='spearman_rho') # Pivot the data: rows = proteins, columns = models, values = spearman_rho
    rank_df = pivot_df.rank(axis=1, ascending=False) # Rank models per protein (higher AUROC = better rank = 1)

    avg_rank = rank_df.mean().sort_values() # Compute average rank per model
    model_order = avg_rank.index.tolist() # THIS ORDERS BY MEAN RANK

    metric = 'spearman_rho' # Possible metrics: 'mean_rank', 'ROC-AUC', 'MCC', 'F1_max', 'F1_mean', 'Accuracy', 'Precision', 'Recall', 'Average_Precision'
    model_order = df.groupby('model')[metric].mean().sort_values(ascending=False).index.tolist() # THIS ORDERS BY MEAN AUROC

    df['model'] = pd.Categorical(df['model'], categories=model_order, ordered=True) # Set categorical order for consistent sorting/plotting
    df = df.sort_values('model') # Sort the DataFrame by the new categorical order
    return df, metric, model_order


Plotting

In [ ]:
def make_plot(df):
    df, metric, model_order = tidy_df(df)
    plt.figure(figsize=(7, 6)) # Plot the boxplot
    plt.axvline(x=0, color='darkgrey', linestyle='--') # add a dotted grey line at y=x
    sns.boxplot(data=df, x=metric, y='model', color='white', linecolor='black', fliersize=0, linewidth=1, showfliers=True) # plot the boxplot
    # sns.violinplot(data= df, x=metric, y='model', color='darkgrey', alpha=1, linewidth=0, inner=None)
    means =  df.groupby('model')[metric].mean() # add dot at mean
    for i, mean in enumerate(means):
        plt.scatter(mean, i, color='lightblue', marker='o', s=50, zorder=10) # parameterize the color? darkred is another
    sns.stripplot(
        data= df,
        x=metric,
        y='model',
        color='black',
        size=3,
        jitter=True,
        alpha=0.3,
        dodge=True
    )
    for i, model in enumerate(model_order): # add text to the plot that has N = number of samples
        n =  df[ df['model'] == model].shape[0]
        break
    print(f"Number of samples: {n}")
    plt.ylabel('') # remove y label
    plt.xlabel('Spearman Rho')
    plt.xlim(-1, 1)
    plt.title('Function-enhancing Mutation Prediction')
    plt.show()

make_plot(func_metrics)

In [ ]:
def make_comparison_plot(df1, df2, color_dict, metric1_name='Metric 1', metric2_name='Metric 2', 
                         title1='Plot 1', title2='Plot 2', figsize=(9, 6),
                         line_a=1,wspace=0.4):
    """
    Create side-by-side ranked box plots with connecting lines showing rank changes.
    
    Parameters:
    -----------
    df1, df2 : pandas.DataFrame
        DataFrames with same models but potentially different rankings
    metric1_name, metric2_name : str
        Names of the metrics being compared
    title1, title2 : str
        Titles for each subplot
    figsize : tuple
        Figure size (width, height)
    wspace : float
        Horizontal spacing between subplots (default 0.4, increase for more space)
    """
    
    # Process both dataframes
    df1_tidy, metric1, order1 = tidy_df(df1.copy())
    df2_tidy, metric2, order2 = tidy_df(df2.copy())
    
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, sharey=False)
    fig.subplots_adjust(wspace=wspace)  # Space between plots for connecting lines
    
    # Plot 1
    ax1.axvline(x=0, color='darkgrey', linestyle='--', alpha=0.5)
    sns.boxplot(data=df1_tidy, x=metric1, y='model', order=order1, 
                color='white', linecolor='black', fliersize=0, 
                linewidth=1, ax=ax1)
    
    means1 = df1_tidy.groupby('model')[metric1].mean()
    for i, model in enumerate(order1):
        ax1.scatter(means1[model], i, color='lightblue', marker='o', s=50, zorder=10)
    
    sns.stripplot(data=df1_tidy, x=metric1, y='model', order=order1,
                  color='black', size=3, jitter=True, alpha=0.3, ax=ax1)
    for label in ax1.get_yticklabels():
        model_name = label.get_text()
        if model_name in color_dict:
            label.set_color(color_dict[model_name])

    ax1.set_ylabel('')
    ax1.set_xlabel(metric1_name,fontsize=14)
    ax1.set_xlim(-1, 1)
    ax1.set_title(title1,fontsize=16)
    ax1.tick_params(right=True, labelright=False, labelsize=12)  # Add ticks on right, no labels
    
    # Plot 2
    ax2.axvline(x=0, color='darkgrey', linestyle='--', alpha=0.5)
    sns.boxplot(data=df2_tidy, x=metric2, y='model', order=order2,
                color='white', linecolor='black', fliersize=0,
                linewidth=1, ax=ax2)
    
    means2 = df2_tidy.groupby('model')[metric2].mean()
    for i, model in enumerate(order2):
        ax2.scatter(means2[model], i, color='lightblue', marker='o', s=50, zorder=10)
    
    sns.stripplot(data=df2_tidy, x=metric2, y='model', order=order2,
                  color='black', size=3, jitter=True, alpha=0.3, ax=ax2)
    
    ax2.set_ylabel('')
    ax2.set_xlabel(metric2_name, fontsize=14)
    ax2.set_xlim(-1, 1)
    ax2.set_title(title2,fontsize=16)
    ax2.yaxis.tick_right()  # Move y-axis ticks to right side
    ax2.yaxis.set_label_position("right")  # Move y-axis label to right side
    for label in ax2.get_yticklabels():
        model_name = label.get_text()
        if model_name in color_dict:
            label.set_color(color_dict[model_name])
    ax2.tick_params(left=True, labelleft=False, labelsize=12)  # Add ticks on left, no labels
    
    # Draw connecting lines between the two plots
    for model in order1:
        if model in order2:
            # Get y-position in each plot (indices)
            y1 = order1.index(model)
            y2 = order2.index(model)
            
            # Convert to figure coordinates
            # Right edge of left plot - use parameterized x position
            point1 = ax1.transData.transform((1.0, y1))
            # Left edge of right plot - use parameterized x position
            point2 = ax2.transData.transform((-1, y2))
            
            # Convert to figure coordinates
            point1_fig = fig.transFigure.inverted().transform(point1)
            point2_fig = fig.transFigure.inverted().transform(point2)
            
            # Calculate rank change for color coding
            rank_change = y2 - y1
            
            # Color: green if improved (moved up = lower index), red if worsened
            if rank_change < 0:
                color = 'black'
            elif rank_change > 0:
                color = 'black'
            else:
                color = 'black'
            
            # Draw line
            line = plt.Line2D([point1_fig[0], point2_fig[0]], 
                            [point1_fig[1], point2_fig[1]],
                            transform=fig.transFigure,
                            color=color, alpha=line_a, linewidth=1.5, zorder=1)
            fig.add_artist(line)
    
    plt.savefig('../images/comparison_plot.svg', format='svg',bbox_inches='tight')
    plt.savefig('../images/comparison_plot.png', dpi=300,bbox_inches='tight')
    plt.show()

In [ ]:
make_comparison_plot(fpdb_metrics, func_metrics, color_dict,
                        metric1_name='Spearman $\\rho$',
                        metric2_name='Spearman $\\rho$',
                        title1='FireprotDB Stability Predictions',
                        title2='ProteinGym Fitness Predictions',
                        wspace=0.25)


### Plot comparing rank of datasets between two models

ProteinGym

In [ ]:
# model0, model1 = 'ESM-3 (hybrid)', 'SaProt 650M'
# model0, model1 = 'ProteinMPNN', 'SolubleMPNN'
def make_intermodel_plot(model0,model1,database='fireprotdb'):
    if database=='fireprotdb':
        group0 = fpdb_metrics[fpdb_metrics['model'] == model0].groupby('protein')['spearman_rho'].mean().dropna()
        group1 = fpdb_metrics[fpdb_metrics['model'] == model1].groupby('protein')['spearman_rho'].mean().dropna()
        plot_title = 'Inter-model FireprotDB Stability Predictions'
        plot_name = '../images/inter-model_comparison_plot_fireprotdb'
    else:
        group0 = func_metrics[func_metrics['model'] == model0].groupby('protein')['spearman_rho'].mean().dropna()
        group1 = func_metrics[func_metrics['model'] == model1].groupby('protein')['spearman_rho'].mean().dropna()
        plot_title = 'Inter-model ProteinGym Fitness Predictions'
        plot_name = '../images/inter-model_comparison_plot_proteingym'

    common_proteins = group0.index.intersection(group1.index)
    group0 = group0.loc[common_proteins]
    group1 = group1.loc[common_proteins]
    categories = common_proteins
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    # Boxplot positions
    pos0, pos1 = 1, 2
    # --- Draw boxplots ---
    ax.boxplot(group0, positions=[pos0], widths=0.4,
            patch_artist=True, boxprops=dict(facecolor='white', alpha=0.3),
            medianprops=dict(color='black', linewidth=2, alpha=0.3))
    ax.boxplot(group1, positions=[pos1], widths=0.4,
            patch_artist=True, boxprops=dict(facecolor='white', alpha=0.3),
            medianprops=dict(color='black', linewidth=2, alpha=0.3))
    # --- Scatter + connecting lines ---
    # Use a colormap to give each category a unique color
    diffs = np.abs(group1 - group0)
    diff_min, diff_max = diffs.min(), diffs.max()
    alpha_min, alpha_max = 0.01, 1
    for cat in categories:
        y0 = float(group0.loc[cat])
        y1 = float(group1.loc[cat])
        alpha = float(np.clip(alpha_min + (alpha_max - alpha_min) * (diffs.loc[cat] - diff_min) / (diff_max - diff_min), alpha_min, alpha_max))
        ax.scatter([pos0], [y0], color='black', zorder=5, alpha=0.3, s=60)
        ax.scatter([pos1], [y1], color='black', zorder=5, alpha=0.3, s=60)
        ax.plot([pos0, pos1], [y0, y1], color='gray', alpha=alpha, linewidth=1.2, zorder=4)

    ax.set_xticks([pos0, pos1])
    tick_colors = [color_dict[model0],color_dict[model1]]
    ax.set_xticklabels([model0, model1], fontsize=14)
    for ticklabel, color in zip(ax.get_xticklabels(), tick_colors):
        ticklabel.set_color(color)

    ax.set_ylabel('Spearman $\\rho$', fontsize=14)
    ax.set_title(plot_title,fontsize=16)
    ax.set_xlim(0.5, 2.5)
    ax.tick_params(labelsize=14)
    ax.axhline(y=0, color='darkgrey', linestyle='--',alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'{plot_name}.svg', format='svg',bbox_inches='tight')
    plt.savefig(f'{plot_name}.png', dpi=300,bbox_inches='tight')
    plt.show()                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

Fireprot DB

In [ ]:
model0, model1 = 'ProteinMPNN', 'SolubleMPNN'

group0 = fpdb_metrics[fpdb_metrics['model'] == model0].groupby('protein')['spearman_rho'].mean().dropna()
group1 = fpdb_metrics[fpdb_metrics['model'] == model1].groupby('protein')['spearman_rho'].mean().dropna()
common_proteins = group0.index.intersection(group1.index)
group0 = group0.loc[common_proteins]
group1 = group1.loc[common_proteins]
categories = common_proteins

fig, ax = plt.subplots(figsize=(5.5, 4.5))

# Boxplot positions
pos0, pos1 = 1, 2

# --- Draw boxplots ---
ax.boxplot(group0, positions=[pos0], widths=0.4,
           patch_artist=True, boxprops=dict(facecolor='white', alpha=0.3),
           medianprops=dict(color='black', linewidth=2, alpha=0.3))

ax.boxplot(group1, positions=[pos1], widths=0.4,
           patch_artist=True, boxprops=dict(facecolor='white', alpha=0.3),
           medianprops=dict(color='black', linewidth=2, alpha=0.3))

# --- Scatter + connecting lines ---
# Use a colormap to give each category a unique color


diffs = np.abs(group1 - group0)
diff_min, diff_max = diffs.min(), diffs.max()
alpha_min, alpha_max = 0.01, 1

for cat in categories:
    y0 = float(group0.loc[cat])
    y1 = float(group1.loc[cat])
    alpha = float(np.clip(alpha_min + (alpha_max - alpha_min) * (diffs.loc[cat] - diff_min) / (diff_max - diff_min), alpha_min, alpha_max))
    ax.scatter([pos0], [y0], color='black', zorder=5, alpha=0.3, s=60)
    ax.scatter([pos1], [y1], color='black', zorder=5, alpha=0.3, s=60)
    ax.plot([pos0, pos1], [y0, y1], color='gray', alpha=alpha, linewidth=1.2, zorder=4)

# --- Labels & legend ---
ax.set_xticks([pos0, pos1])
ax.set_xticklabels([model0, model1],fontsize=14,color='tab:red')
ax.set_ylabel('Spearman $\\rho$', fontsize=14)
ax.set_title('Inter-model FireprotDB Stability Predictions',fontsize=16)
ax.set_xlim(0.5, 2.5)
ax.tick_params(labelsize=14)
ax.axhline(y=0, color='darkgrey', linestyle='--',alpha=0.5)
plt.tight_layout()
plt.savefig('../images/inter-model_comparison_plot_fireprotdb.svg', format='svg',bbox_inches='tight')
plt.savefig('../images/inter-model_comparison_plot_fireprotdb.png', dpi=300,bbox_inches='tight')
plt.show()                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             